# Fork A: convergence-matched data-efficiency sweep

Kills the under-convergence confound in the data-efficiency null: instead of the light
`--epochs-phase2 40` budget (which left the physics arm at negative test R^2), **both arms
train to their validation plateau** at each fraction/seed. Physics arm trains **first** (it is
the at-risk arm), then the matched DirectGNN control. Cheapest fraction first, seed-major (a
complete seed-42 curve banks before 43/44), wall-clock guarded, saved after every run, with an
inline PLATEAUED / CAP-HIT audit per physics run.

**Before running:**
1. Settings -> Accelerator -> **GPU T4 x2** (P100 = sm_60 is too old for the installed PyTorch; the run guard will abort on it).
2. Settings -> Internet -> **On** (to clone the repo + pip install).
3. **Add Data** -> attach your processed-split dataset (must contain `train.csv`, `val.csv`, `test.csv`, and `crystal_train.csv`). Use the SAME split as the headline run -- do NOT regenerate it (the seeded scaffold split is unstable across pipeline versions and a new split orphans comparability).
4. **Run the smoke cell first** (~2-3 min) to verify the wiring, then the real run.

In [ ]:
REPO_URL = 'https://github.com/doctawho42/tgnn-solv.git'
BRANCH   = 'sigma-grounded-cosmosac'
REPO     = '/kaggle/working/tgnn-solv'
DEADLINE_HOURS = 8.0     # leave ~1h under the Kaggle session limit for packaging
BATCH, WORKERS = 256, 4  # config default batch 64 starves the GPU

In [ ]:
import os, subprocess
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',REPO_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','--depth','1','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard',f'origin/{BRANCH}'], check=True)
head = subprocess.run(['git','-C',REPO,'rev-parse','--short','HEAD'], capture_output=True, text=True).stdout.strip()
print('repo at', head)

In [ ]:
import subprocess
subprocess.run(['pip','install','-q','-e',REPO], check=True)
# Kaggle ships torch + CUDA; make sure the graph/chem deps are present
subprocess.run(['pip','install','-q','torch-geometric','rdkit'], check=False)
print('deps installed')

In [ ]:
# Stage the attached processed-split dataset into the repo's expected paths.
import glob, shutil
from pathlib import Path
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None
targets = {
    'train.csv': f'{REPO}/notebooks/data/processed/train.csv',
    'val.csv':   f'{REPO}/notebooks/data/processed/val.csv',
    'test.csv':  f'{REPO}/notebooks/data/processed/test.csv',
    'crystal_train.csv': f'{REPO}/notebooks/data/processed_crystal_aux_stream/crystal_train.csv',
}
for name, dst in targets.items():
    src = find(name)
    assert src, f'MISSING {name} under /kaggle/input -- attach your processed-data dataset (Add Data)'
    Path(dst).parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, dst)
    print(f'staged {name:20s} <- {src}')

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU visible -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))

### Smoke test first (~2-3 min)
One tiny point (seed 42, frac 0.05, 1-2 epochs/arm) that exercises the WHOLE pipeline: subsample -> train physics (train.py) -> export -> plateau audit -> train DirectGNN control -> export. Verify the logs look right and both arms report a metric **before** spending the 9h. Physics trains first (it is the at-risk arm).

In [ ]:
import os, subprocess, json
env = {**os.environ, 'KMP_DUPLICATE_LIB_OK':'TRUE', 'PYTHONPATH':f'{REPO}/src', 'PYTHONUNBUFFERED':'1'}
subprocess.run(['python', f'{REPO}/scripts/cloud/kaggle_run.py',
                '--do','dataeff_converged','--smoke','--out','/kaggle/working/smoke','--device','cuda',
                '--batch-size',str(BATCH),'--workers',str(WORKERS)], cwd=REPO, env=env, check=False)
print('\n--- smoke result ---')
sp = '/kaggle/working/smoke/de_converged/dataeff_converged.json'
print(open(sp).read() if os.path.exists(sp) else 'NO RESULT -- inspect the log above before the real run')

### Real run
Only after the smoke result shows both arms with a metric. Streams live, saves after every run, packages partial results on timeout.

In [ ]:
import os, subprocess
env = {**os.environ, 'KMP_DUPLICATE_LIB_OK':'TRUE', 'PYTHONPATH':f'{REPO}/src', 'PYTHONUNBUFFERED':'1'}
cmd = ['python', f'{REPO}/scripts/cloud/kaggle_run.py',
       '--do','dataeff_converged','--out','/kaggle/working/results','--device','cuda',
       '--batch-size',str(BATCH),'--workers',str(WORKERS),'--deadline-hours',str(DEADLINE_HOURS)]
subprocess.run(cmd, cwd=REPO, env=env, check=False)  # check=False: package partial results even on timeout

In [ ]:
# Package results + print the banked curve (download the zip from the Output panel).
import os, json, shutil
shutil.make_archive('/kaggle/working/dataeff_converged_results', 'zip', '/kaggle/working/results')
print('zipped -> /kaggle/working/dataeff_converged_results.zip\n')
p = '/kaggle/working/results/de_converged/dataeff_converged.json'
if os.path.exists(p):
    for r in json.load(open(p)):
        if r.get('status') == 'deferred_budget':
            print(f"  s{r['seed']} f{r['frac']}: DEFERRED (out of budget)")
        else:
            d, ph = r.get('direct', {}), r.get('physics', {})
            print(f"  s{r['seed']} f{r['frac']}: direct R2={d.get('r2')} MAE={d.get('mae')} | "
                  f"physics R2={ph.get('r2')} MAE={ph.get('mae')} | dMAE={r.get('delta_mae')}")
else:
    print('no results json yet -- check the run cell output above')